In [13]:
import collections

k = 4
X = 2
match = 3
mismatch = -3
gap = -2

database = "CTAGGATCCAGGCATACGA"
query = "GGATCCATTCATTA"

def build_index(database, k):
    index = collections.defaultdict(list)
    for pos in range(len(database) - k + 1):
        kmer = database[pos:pos + k]
        index[kmer].append(pos)
    return dict(index)

def get_score(a, b, match, mismatch):
    return match if a == b else mismatch

def extend_direction(q_seq, d_seq, seed_score, X, match, mismatch, gap, is_left):
    scur = seed_score
    smax = seed_score
    max_pos = 0
    history = [(scur, 0 if is_left else len(q_seq) - len(d_seq), 0 if is_left else len(q_seq) - len(d_seq), ''.join(q_seq[::-1] if is_left else q_seq), ''.join(d_seq[::-1] if is_left else d_seq))]  # Упрощённо для seed
    l = min(len(q_seq), len(d_seq))
    p = 0
    pos_q = -1 if is_left else len(q_seq)
    pos_d = -1 if is_left else len(d_seq)
    while p < l:
        a = q_seq[p]
        b = d_seq[p]
        s = get_score(a, b, match, mismatch)
        diag = scur + s
        gap_q = scur + gap
        gap_d = scur + gap
        scur = max(diag, gap_q, gap_d)
        pos_q += -1 if is_left else 1
        pos_d += -1 if is_left else 1
        history.append((scur, pos_q, pos_d, a, b))
        if scur > smax:
            smax = scur
            max_pos = p + 1
        if smax - scur >= X:
            break
        p += 1
    expanded_q = q_seq[0:max_pos]
    expanded_d = d_seq[0:max_pos]
    if is_left:
        expanded_q = expanded_q[::-1]
        expanded_d = expanded_d[::-1]
    return smax, history, expanded_q, expanded_d

def seed_and_extend(query, database, k, X, match, mismatch, gap):
    index = build_index(database, k)
    print("Индекс БД:")
    for kmer, positions in index.items():
        print(f" {kmer}: {positions}")
    seeds = []
    for q_pos in range(len(query) - k + 1):
        kmer = query[q_pos:q_pos + k]
        if kmer in index:
            for d_pos in index[kmer]:
                seeds.append((q_pos, d_pos))
    print("Найденные seed:")
    for s in seeds:
        print(f" {s}")
    best_score = -float('inf')
    best_seed = None
    best_history_left = best_history_right = None
    best_ext_left_q = best_ext_left_d = None
    best_ext_right_q = best_ext_right_d = None
    for sq, sd in seeds:
        print(f"Обработка seed (query {sq}, db {sd})")
        seed_score = k * match
        q_left = query[0:sq][::-1]
        d_left = database[0:sd][::-1]
        left_smax, hist_left, left_q, left_d = extend_direction(q_left, d_left, seed_score, X, match, mismatch, gap, True)
        print(" Расширение влево:")
        for step in hist_left:
            print(f" S_cur={step[0]}, ({step[1]},{step[2]}) '{step[3]}'|'{step[4]}'")
        print(f" Левый Smax = {left_smax}")
        q_right = query[sq + k:]
        d_right = database[sd + k:]
        right_smax, hist_right, right_q, right_d = extend_direction(q_right, d_right, seed_score, X, match, mismatch, gap, False)
        print(" Расширение вправо:")
        for step in hist_right:
            print(f" S_cur={step[0]}, ({step[1]},{step[2]}) '{step[3]}'|'{step[4]}'")
        print(f" Правый Smax = {right_smax}")
        total = left_smax + right_smax - seed_score
        print(f" Общий счёт = {total}")
        if total > best_score:
            best_score = total
            best_seed = (sq, sd)
            best_history_left = hist_left
            best_history_right = hist_right
            best_ext_left_q = left_q
            best_ext_left_d = left_d
            best_ext_right_q = right_q
            best_ext_right_d = right_d
    print("Результат:")
    print(f"Лучшее seed: позиции query={best_seed[0]}, database={best_seed[1]}, k-мер = '{query[best_seed[0]:best_seed[0]+k]}'")
    print(f"Итоговый S_max = {best_score}")
    print("История расширения влево (Scur, позиция Q, позиция D, символы):")
    for step in best_history_left:
        print(f" S_cur={step[0]}, ({step[1]},{step[2]}) '{step[3]}'|'{step[4]}'")
    print("История расширения вправо:")
    for step in best_history_right:
        print(f" Scur={step[0]}, ({step[1]},{step[2]}) '{step[3]}'|'{step[4]}'")
    full_q = best_ext_left_q + query[best_seed[0]:best_seed[0] + k] + best_ext_right_q
    full_d = best_ext_left_d + database[best_seed[1]:best_seed[1] + k] + best_ext_right_d
    
    print("Итоговое выравнивание:")
    print(f"Q: {' '.join(full_q)}")
    print(f"D: {' '.join(full_d)}")

seed_and_extend(query, database, k, X, match, mismatch, gap)

Индекс БД:
 CTAG: [0]
 TAGG: [1]
 AGGA: [2]
 GGAT: [3]
 GATC: [4]
 ATCC: [5]
 TCCA: [6]
 CCAG: [7]
 CAGG: [8]
 AGGC: [9]
 GGCA: [10]
 GCAT: [11]
 CATA: [12]
 ATAC: [13]
 TACG: [14]
 ACGA: [15]
Найденные seed:
 (0, 3)
 (1, 4)
 (2, 5)
 (3, 6)
Обработка seed (query 0, db 3)
 Расширение влево:
 S_cur=12, (0,0) ''|'CTA'
 Левый Smax = 12
 Расширение вправо:
 S_cur=12, (-2,-2) 'CCATTCATTA'|'CCAGGCATACGA'
 S_cur=15, (11,13) 'C'|'C'
 S_cur=18, (12,14) 'C'|'C'
 S_cur=21, (13,15) 'A'|'A'
 S_cur=19, (14,16) 'T'|'G'
 Правый Smax = 21
 Общий счёт = 21
Обработка seed (query 1, db 4)
 Расширение влево:
 S_cur=12, (0,0) 'G'|'CTAG'
 S_cur=15, (-2,-2) 'G'|'G'
 Левый Smax = 15
 Расширение вправо:
 S_cur=12, (-2,-2) 'CATTCATTA'|'CAGGCATACGA'
 S_cur=15, (10,12) 'C'|'C'
 S_cur=18, (11,13) 'A'|'A'
 S_cur=16, (12,14) 'T'|'G'
 Правый Smax = 18
 Общий счёт = 21
Обработка seed (query 2, db 5)
 Расширение влево:
 S_cur=12, (0,0) 'GG'|'CTAGG'
 S_cur=15, (-2,-2) 'G'|'G'
 S_cur=18, (-3,-3) 'G'|'G'
 Левый Smax = 18
 Р